In [2]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from transformers import TrainingArguments, Trainer
import evaluate
import collections
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [4]:
dataset = load_dataset("squad")

print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome

In [5]:
model_checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [6]:
max_length = 384
stride = 128

def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

In [7]:
train_dataset = dataset["train"].select(range(40000))
validation_dataset = dataset["validation"].select(range(3000))
tokenized_train = train_dataset.map(
    preprocess_training_examples,
    batched=True,
    remove_columns=train_dataset.column_names
)

print(tokenized_train)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 40383
})


In [8]:
training_args = TrainingArguments(
    output_dir="./bert-squad-results-clean",
    eval_strategy="no",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train
)

trainer.train()

Step,Training Loss
100,4.428066
200,2.903428
300,2.372833
400,2.068737
500,1.871110
600,1.685282
700,1.748282
800,1.660455
900,1.632318
1000,1.420045


TrainOutput(global_step=15144, training_loss=0.8493591988458246, metrics={'train_runtime': 2657.0058, 'train_samples_per_second': 45.596, 'train_steps_per_second': 5.7, 'total_flos': 2.374188058635725e+16, 'train_loss': 0.8493591988458246, 'epoch': 3.0})

In [9]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offsets = inputs["offset_mapping"][i]

        inputs["offset_mapping"][i] = [
            offset if sequence_ids[k] == 1 else None
            for k, offset in enumerate(offsets)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [10]:
tokenized_validation = validation_dataset.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=validation_dataset.column_names
)

print(tokenized_validation)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 3037
})


In [11]:
raw_predictions = trainer.predict(tokenized_validation)

start_logits, end_logits = raw_predictions.predictions

print(start_logits.shape)
print(end_logits.shape)

(3037, 384)
(3037, 384)


In [12]:
metric = evaluate.load("squad")

n_best = 20
max_answer_length = 30

examples = validation_dataset
features = tokenized_validation

example_to_features = collections.defaultdict(list)

for idx, feature in enumerate(features):
    example_to_features[feature["example_id"]].append(idx)

predicted_answers = []

for example in examples:
    example_id = example["id"]
    context = example["context"]
    answers = []

    for feature_index in example_to_features[example_id]:
        start_logit = start_logits[feature_index]
        end_logit = end_logits[feature_index]
        offsets = features[feature_index]["offset_mapping"]

        start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
        end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()

        for start_index in start_indexes:
            for end_index in end_indexes:
                if offsets[start_index] is None or offsets[end_index] is None:
                    continue

                if end_index < start_index:
                    continue

                if end_index - start_index + 1 > max_answer_length:
                    continue

                start_char = offsets[start_index][0]
                end_char = offsets[end_index][1]

                answer_text = context[start_char:end_char]
                score = start_logit[start_index] + end_logit[end_index]

                answers.append({
                    "text": answer_text,
                    "logit_score": score
                })

    if len(answers) > 0:
        best_answer = max(answers, key=lambda x: x["logit_score"])
        predicted_answers.append({
            "id": example_id,
            "prediction_text": best_answer["text"]
        })
    else:
        predicted_answers.append({
            "id": example_id,
            "prediction_text": ""
        })

theoretical_answers = [
    {
        "id": ex["id"],
        "answers": ex["answers"]
    }
    for ex in examples
]

results = metric.compute(
    predictions=predicted_answers,
    references=theoretical_answers
)

results

{'exact_match': 77.46666666666667, 'f1': 85.0156626785667}

In [13]:
from transformers import pipeline
import torch
model = model.to(device)
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

In [14]:
context = """
The Eiffel Tower is located in Paris, France.
It was completed in 1889 and is one of the most famous landmarks in the world.
"""

question = "Where is the Eiffel Tower located?"

result = qa_pipeline(question=question, context=context)

print("Question:", question)
print("Answer:", result["answer"])
print("Confidence:", result["score"])

Question: Where is the Eiffel Tower located?
Answer: Paris, France
Confidence: 0.9402308464050293


In [15]:
for i in range(5):
    ex = validation_dataset[i]

    result = qa_pipeline(
        question=ex["question"],
        context=ex["context"]
    )

    print("=" * 80)
    print("Question:", ex["question"])
    print("Correct answer:", ex["answers"]["text"][0])
    print("Model answer:", result["answer"])
    print("Confidence:", result["score"])

Question: Which NFL team represented the AFC at Super Bowl 50?
Correct answer: Denver Broncos
Model answer: Denver Broncos
Confidence: 0.9559071660041809
Question: Which NFL team represented the NFC at Super Bowl 50?
Correct answer: Carolina Panthers
Model answer: Carolina Panthers
Confidence: 0.8575058579444885
Question: Where did Super Bowl 50 take place?
Correct answer: Santa Clara, California
Model answer: Levi's Stadium in the San Francisco Bay Area at Santa Clara, California
Confidence: 0.3848976492881775
Question: Which NFL team won Super Bowl 50?
Correct answer: Denver Broncos
Model answer: Denver Broncos
Confidence: 0.8042441010475159
Question: What color was used to emphasize the 50th anniversary of the Super Bowl?
Correct answer: gold
Model answer: golden anniversary" with various gold
Confidence: 0.5071197748184204


In [21]:
import ipywidgets as widgets
from IPython.display import display, clear_output
context_box = widgets.Textarea(
    value="""The Eiffel Tower is located in Paris, France. It was completed in 1889 and is one of the most famous landmarks in the world.""",
    placeholder="Paste your paragraph here...",
    description="Paragraph:",
    layout=widgets.Layout(width="100%", height="180px")
)

questions_box = widgets.Textarea(
    value="""Where is the Eiffel Tower located?
When was the Eiffel Tower completed?
What is one of the most famous landmarks in the world?""",
    placeholder="Write one question per line...",
    description="Questions:",
    layout=widgets.Layout(width="100%", height="120px")
)

run_button = widgets.Button(
    description="Get Answers",
    button_style="success"
)

output_area = widgets.Output()

def answer_questions(button):
    with output_area:
        clear_output()

        context = context_box.value.strip()
        questions = [q.strip() for q in questions_box.value.split("\n") if q.strip()]

        if not context:
            print("Please enter a paragraph.")
            return

        if not questions:
            print("Please enter at least one question.")
            return

        for i, question in enumerate(questions, 1):
            result = qa_pipeline(question=question, context=context)



            print("=" * 80)
            print(f"Question {i}: {question}")
            print(f"Answer: {result['answer']}")
            print(f"Confidence: {result['score']:.4f}")

run_button.on_click(answer_questions)

display(context_box, questions_box, run_button, output_area)

Textarea(value='The Eiffel Tower is located in Paris, France. It was completed in 1889 and is one of the most …

Textarea(value='Where is the Eiffel Tower located?\nWhen was the Eiffel Tower completed?\nWhat is one of the m…

Button(button_style='success', description='Get Answers', style=ButtonStyle())

Output()

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
model.save_pretrained("/content/drive/MyDrive/bert_qa_model_40k")
tokenizer.save_pretrained("/content/drive/MyDrive/bert_qa_model_40k")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/bert_qa_model_40k/tokenizer_config.json',
 '/content/drive/MyDrive/bert_qa_model_40k/tokenizer.json')

In [19]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import torch
model_path = "/content/drive/MyDrive/bert_qa_model_40k"
model = AutoModelForQuestionAnswering.from_pretrained(model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]